In [ ]:
import pandas as pd
import numpy as np
import math
import text_utils
import heapq

In [ ]:
df = pd.read_csv("spam.csv", encoding='latin-1')
print(df)


# v1 = label (spam/ham), v2 = text message
y = df['v1'].values  # labels
x = df['v2'].values  # raw text messages

rows = len(x)

# Shuffle and split
np.random.seed(42)  
shuffle_idx = np.random.permutation(rows)
train_size = int(0.8 * rows)

train_idx = shuffle_idx[:train_size]
test_idx = shuffle_idx[train_size:]

x_train = x[train_idx]  
x_test = x[test_idx]
y_train = y[train_idx]
y_test = y[test_idx]

In [ ]:
#A map that stores words in the format word:index
vocab_list = {}
#len(vocab_list)
vocab_size = 1000

#A simple tokenization of the text

#Creating tokens for training and tetsing which will later be vectorized
train_tokens = [text_utils.tokenize(x_train[i]) for i in range(len(x_train))]
test_tokens = [text_utils.tokenize(x_test[j]) for j in range(len(x_test))]

vocab_list = text_utils.build_vocab_list(vocab_size, x_train, train_tokens)

#Creating vectorized inputs for each training example in x_train
vectorized_inputs = [[0] * vocab_size for _ in range(len(x_train))]

vectorized_inputs = text_utils.vectorize(x_train, vocab_size, train_tokens, vocab_list, True)

#Calculating the class ratios
class_ratio_map = text_utils.calculate_class_ratio(y_train)

In [ ]:
#This will store the phi parameter used in the Multinomial Distribution as (k, j) where k corresponds to the class and j corresponds to the feature
phi_dict = {}
#phi j,k = (occurences of a feature exisiting given a specific class)/(number of examples of that specific class)
def calculate_word_count(vectorized_x, training_y):
    res_dict = {}
    for i in range(len(vectorized_x)):
        for j in range(len(vectorized_x[0])):
            if training_y[i] not in res_dict:
                res_dict[training_y[i]] = vectorized_x[i][j]
            else:
                res_dict[training_y[i]] += vectorized_x[i][j]
    
    return res_dict

def calculate_phi_dict(vectorized_x, training_y):
    feature_number = 0
    total_words = calculate_word_count(vectorized_x, training_y)
    for i in range(len(vectorized_x[0])):
        auxilary_dict = {element:0 for element in class_ratio_map}
        for j in range(len(vectorized_x)):
            auxilary_dict[training_y[j]] += vectorized_x[j][i]
    
        for elt in auxilary_dict:
            #Laplace smoothing to ensure log(0) probabilities never appear
            phi_dict[(elt, feature_number)] = (auxilary_dict[elt] + 1)/(total_words[elt] + vocab_size)
        
        feature_number += 1

calculate_phi_dict(vectorized_inputs, y_train)

def predict_one(test_vector, training_y):
    heap = []
    heapq.heapify_max(heap)
    for element in class_ratio_map:
        s = 0
        for i in range(len(test_vector)):
            s += test_vector[i] * math.log(phi_dict[(element, i)])
        heapq.heappush_max(heap, (math.log(class_ratio_map[element]/len(training_y)) + s, element))
    
    return heap[0][1]


#Vectorizing test inputs
vectorized_test_inputs = [[0] * 1000 for _ in range(len(x_test))]
vectorized_test_inputs = text_utils.vectorize(x_test, vocab_size, test_tokens, vocab_list, True)

#Making predictions
def make_all_predictions(test_x, test_y, training_y):
    accuracy = 0
    ham_count = 0
    spam_count = 0
    for i in range(len(test_x)):
        prediction = predict_one(test_x[i], training_y)
        if(prediction == test_y[i]):
            accuracy += 1

        if prediction == 'ham':
            ham_count += 1
        else:
            spam_count += 1
    
    print(accuracy/len(test_x))

make_all_predictions(vectorized_test_inputs, y_test, y_train)

    